# Part 2: Streaming application using Spark Structured Streaming  
In this task, you will implement Spark Structured Streaming to consume the data from task 1 and perform a prediction.    
Important: 
- This task uses PySpark Structured Streaming with PySpark Dataframe APIs and PySpark ML.
- You also need your pipeline model from A2A to make predictions and persist the results.
- Note for the prediction related to event time: in a real scenario, you should use accident_ts as the event time; however, since we are simulating streaming and your model was trained with the time column as a feature, you can choose to use the time column or accident_ts.


1. Write code to create a SparkSession, which 1) uses four cores with a proper application name; 2) uses the UK/London timezone; 3) ensures a checkpoint location has been set.

In [ ]:
from pyspark.sql import SparkSession

# Create a Spark session 
spark = (
    SparkSession.builder
    .master("local[4]")
    .appName("A2B_Accident_Streaming_Prediction") #Set application name
    .config("spark.sql.session.timeZone", "Europe/London") # Set timezone for timestamp processing
    .config("spark.sql.streaming.checkpointLocation", "./checkpoint/a2b") #Set checkpoint
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0" # Add Kafka package for Spark
    )
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")


2. Write code to define the data schema for the data files. Load the static datasets into data frames. (You can reuse your code from 2A.) In a car accident, we collection streaming information like the realtime road condition, but the vehicle information is static and can be read from the vehicle registration database.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
#Define Schema
vehicle_schema = StructType([
    StructField("collision_index", StringType(), True),
    StructField("vehicle_reference", IntegerType(), True),
    StructField("vehicle_type", StringType(), True),
    StructField("vehicle_manoeuvre", StringType(), True),
    StructField("junction_location", StringType(), True),
    StructField("skidding_and_overturning", StringType(), True),
    StructField("hit_object_in_carriageway", StringType(), True),
    StructField("hit_object_off_carriageway", IntegerType(), True),
    StructField("first_point_of_impact", StringType(), True),
    StructField("sex_of_driver", StringType(), True),
    StructField("age_of_driver", IntegerType(), True),
    StructField("engine_capacity_cc", IntegerType(), True),
    StructField("propulsion_code", IntegerType(), True),
    StructField("age_of_vehicle", IntegerType(), True)
])
df_vehicle = spark.read.csv(
    "vehicle.csv",
    header=True,
    schema=vehicle_schema
)

3. Using the Kafka topic from the producer in Task 1, ingest the streaming data into Spark Streaming, assuming all data comes in the String format. Except for the 'accident_ts' column, you shall receive it as a numeric type. Then, the data frames should be transformed into the appropriate types.

In [ ]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType, LongType

kafka_topic = "accident_stream"

# Get column names from streaming_collision.csv
collision_columns = (
    spark.read
    .option("header", True)
    .csv("streaming_collision.csv")
    .columns
)

# Define schema
stream_schema = StructType()
for column_name in collision_columns:
    stream_schema.add(StructField(column_name, StringType(), True))
stream_schema.add(StructField("accident_ts", LongType(), True))

# Read stream from Kafka
raw_kafka_df = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("subscribe", kafka_topic)
    .option("startingOffsets", "latest") # Read only new messages
    .load()
)

# Convert Kafka message value from JSON string into DataFrame columns
accident_stream_df = (
    raw_kafka_df
    .selectExpr("CAST(value AS STRING) AS json_value")  # Convert message value to string
    .select(from_json(col("json_value"), stream_schema).alias("data")) # Parse JSON using schema
    .select("data.*") # Expand parsed data into separate columns
)
# Print the schema of the streaming DataFrame
accident_stream_df.printSchema()

4. Use a watermark on accident_ts. If data points are received 30 seconds late, discard the data. (note: in a local environment like your laptop, late arrival or delayed processing may never happen.)

In [ ]:
from pyspark.sql.functions import col, from_unixtime

# Convert accident_ts into timestamp format for watermark
accident_stream_df = accident_stream_df.withColumn(
    "accident_time",
    from_unixtime(col("accident_ts")).cast("timestamp")
)

# Allow Spark to handle records that arrive up to 30 seconds late
accident_stream_df = accident_stream_df.withWatermark(
    "accident_time",
    "30 seconds"
)

accident_stream_df.printSchema()

5. Perform the necessary transformation you used in A2A. (note: every student may have used different features, feel free to reuse the code you have written in A2A. If you built an end-to-end pipeline, you can ignore this task.) 

In [ ]:
from pyspark.sql import functions as F
#Aggregate vehicle table
vehicle_agg_df = (
    df_vehicle
    .groupBy("collision_index")
    .agg(
        F.countDistinct("vehicle_reference").alias("num_vehicles"),

        # vehicle type information
        F.max(F.when(F.col("vehicle_type").isin(9,108,109), True).otherwise(False)).alias("has_car"),
        F.max(F.when(F.col("vehicle_type").isin(2, 3, 4, 5, 97, 103, 104, 105, 106), True).otherwise(False)).alias("has_motorcycle"),
        F.max(F.when(F.col("vehicle_type") == 1, True).otherwise(False)).alias("has_pedal_cycle"),
        F.max(F.when(F.col("vehicle_type").isin(10, 11, 110), True).otherwise(False)).alias("has_bus_or_minibus"),
        F.max(F.when(F.col("vehicle_type").isin(19, 20, 21, 98, 113), True).otherwise(False)).alias("has_goods_vehicle"),

        # behaviour/ crash indicators
        F.max(F.when(F.col("skidding_and_overturning") > 0, True).otherwise(False)).alias("has_skidding_or_overturning"),
        F.max(F.when(F.col("hit_object_in_carriageway") > 0, True).otherwise(False)).alias("hit_object_in_carriageway_flag"),
        F.max(F.when(F.col("hit_object_off_carriageway") > 0, True).otherwise(False)).alias("hit_object_off_carriageway_flag"),
    )
)


In [ ]:
# Join streaming accident data with static aggregated vehicle data
accident_vehicle_stream_df = accident_stream_df.join(
    vehicle_agg_df,
    on="collision_index",
    how="left"
)

accident_vehicle_stream_df.printSchema()

In [ ]:
from pyspark.sql.functions import col, hour, to_timestamp, when


accident_vehicle_stream_df  = accident_vehicle_stream_df.withColumn(
    "Hour",
    hour(to_timestamp(col("time"), "HH:mm"))
)

# Create Peak_Traffic feature
accident_vehicle_stream_df = accident_vehicle_stream_df.withColumn(
    "Peak_Traffic",
    when(
        ((col("Hour") >= 7) & (col("Hour") <= 9)) |
        ((col("Hour") >= 16) & (col("Hour") <= 18)),
        "Peak"
    ).otherwise("Off-peak")
)

# create another feature
from pyspark.sql.functions import col, to_date, dayofweek, when

# Convert date column to date type
accident_vehicle_stream_df = accident_vehicle_stream_df.withColumn(
    "date_parsed",
    to_date(col("date"), "dd/MM/yyyy")
)

# Create is_weekend feature
# Sunday is 1, Saturday is 7
accident_vehicle_stream_df  = accident_vehicle_stream_df.withColumn(
    "is_weekend",
    when(dayofweek(col("date_parsed")).isin(1, 7), True).otherwise(False)
)


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

# Define feature columns based on the trained model
categorical_columns = [
    "Peak_Traffic",
    "road_type",
    "junction_detail",
    "carriageway_hazards",
    "light_conditions",
    "weather_conditions",
    "road_surface_conditions",
    "urban_or_rural_area"
]

numeric_columns = [
    "speed_limit",
    "num_vehicles"
]

binary_columns = [
    "has_car",
    "has_motorcycle",
    "has_pedal_cycle",
    "has_bus_or_minibus",
    "has_goods_vehicle",
    "has_skidding_or_overturning",
    "hit_object_in_carriageway_flag",
    "hit_object_off_carriageway_flag",
    "is_weekend"
]

# Cast categorical columns to string
for c in categorical_columns:
    accident_vehicle_stream_df = accident_vehicle_stream_df.withColumn(
        c,
        col(c).cast("string")
    )

# Cast numeric columns to double
for c in numeric_columns:
    accident_vehicle_stream_df = accident_vehicle_stream_df.withColumn(
        c,
        col(c).cast("double")
    )

# Cast binary columns to boolean
for c in binary_columns:
    accident_vehicle_stream_df = accident_vehicle_stream_df.withColumn(
        c,
        col(c).cast("boolean")
    )
    


In [ ]:
# Group speed_limit values into standard speed categories
accident_vehicle_stream_df = accident_vehicle_stream_df.withColumn(
    "speed_limit",
     F.when(F.col("speed_limit") <= 20, 20)
     .when((F.col("speed_limit") > 20) & (F.col("speed_limit") <= 30), 30)
     .when((F.col("speed_limit") > 30) & (F.col("speed_limit") <= 40), 40)
     .when((F.col("speed_limit") > 40) & (F.col("speed_limit") <= 50), 50)
     .when((F.col("speed_limit") > 50) & (F.col("speed_limit") <= 60), 60)
     .when((F.col("speed_limit") > 60) & (F.col("speed_limit") <= 70), 70)
     .otherwise(70)
)
    


In [ ]:
# Keep only valid accident records
clean_condition = (
    (F.col("road_type") != -1) &
    (F.col("junction_detail") != -1) &
    (F.col("light_conditions") != -1) &
    (F.col("weather_conditions") != -1) &
    (F.col("carriageway_hazards") != -1) &
    (F.col("road_surface_conditions") != -1) &
    (F.col("urban_or_rural_area") != 3)
)

accident_vehicle_stream_df = accident_vehicle_stream_df.filter(clean_condition)


In [ ]:
accident_vehicle_stream_df.printSchema()

6. Load your pipeline model and perform the following aggregations:  
a) Make predictions and print high-severity accidents (>7) every 5 seconds.   
b) Every 10 seconds, print the total number of accidents for each severity.  
c) Every 30 seconds, for each local district with data, print the total number of low (1-3), medium (4-6) and high severity (7-10) accidents.  

In [ ]:
from pyspark.ml import PipelineModel
from pyspark.sql.functions import col, count, window, when

# Load pipeline model from A2A
model_path = "best_regression_model"   # change to your saved model path
pipeline_model = PipelineModel.load(model_path)

# Make predictions
prediction_df = pipeline_model.transform(accident_vehicle_stream_df)
prediction_df.printSchema()


### a) Make predictions and print high-severity accidents (>7) every 5 seconds.

In [ ]:
# 6a
import time
# Convert prediction to severity number
prediction_df = prediction_df.withColumn(
    "severity_rating",
    F.round(col("prediction")).cast("int")
)

# Print high-severity accidents (>7) every 5 seconds
high_severity_df = (
    prediction_df
    .filter(col("severity_rating") > 7)
    .select(
        "collision_index",
        "accident_time",
        "area",
        "longitude",
        "latitude",
        "severity_rating"
    )
)
# Write high-severity accidents to a memory table every 5 seconds
high_severity_query = (
    high_severity_df
    .writeStream
    .format("memory") # Store streaming output in memory
    .queryName("high_severity_table") # Name of memory table
    .outputMode("append") # Add new records only
    .option("checkpointLocation", f"./checkpoint/high_severity_{int(time.time())}") # Save streaming progress
    .trigger(processingTime="5 seconds") # Update every 5 seconds
    .start()
)

### b) Every 10 seconds, print the total number of accidents for each severity.

In [ ]:
# 6b

# Count total number of accidents for each severity
severity_count_df = (
    prediction_df
    .groupBy("severity_rating")
    .count()
    .orderBy("severity_rating")
)

# Write severity count results to a memory table every 10 seconds
severity_count_query = (
    severity_count_df
    .writeStream
    .format("memory") # Store streaming output in memory
    .queryName("severity_count_table") # Name of the memory table
    .outputMode("complete") # Show the full updated result each time
    .option("checkpointLocation", f"./checkpoint/severity_count_{int(time.time())}") # Save streaming progress
    .trigger(processingTime="10 seconds") # Update every 10 seconds
    .start()
)

### c) Every 30 seconds, for each local district with data, print the total number of low (1-3), medium (4-6) and high severity (7-10) accidents.

In [ ]:
# 6c
from pyspark.sql import functions as F
from pyspark.sql.functions import col, window
import time

# Create severity group: low, medium, high
severity_group_df = prediction_df.withColumn(
    "severity_group",
    F.when((col("severity_rating") >= 1) & (col("severity_rating") <= 3), "low")
     .when((col("severity_rating") >= 4) & (col("severity_rating") <= 6), "medium")
     .when((col("severity_rating") >= 7) & (col("severity_rating") <= 10), "high")
)

# Count accidents by local district and severity group every 30 seconds
district_severity_df = (
    severity_group_df
    .filter(col("area").isNotNull())
    .groupBy(
        window(col("accident_time"), "30 seconds"),
        col("area"),
        col("severity_group")
    )
    .count()
)

# Write district severity results to a memory table every 30 seconds
district_severity_query = (
    district_severity_df
    .writeStream
    .format("memory") # Store streaming output in memory
    .queryName("district_severity_table") # Name of the memory table
    .outputMode("complete") # Show the full updated result each time
    .option("checkpointLocation", f"./checkpoint/district_severity_{int(time.time())}") # Save streaming progress
    .trigger(processingTime="30 seconds")  # Update every 30 seconds
    .start()
)

7. Save the data from 6 to Parquet files as streams. (Hint: Parquet files support streaming writing/reading. The file should keep updating while new batches arrive.)

In [ ]:
# 7a(save 6a)
import time

# Save high-severity accident records to Parquet every 5 seconds
high_severity_parquet_query = (
    high_severity_df
    .writeStream
    .format("parquet")
    .outputMode("append")
    .option("path", "./parquet_output/6a_high_severity")
    .option("checkpointLocation", f"./checkpoint/parquet_6a_high_severity_{int(time.time())}")
    .trigger(processingTime="5 seconds")
    .start()
)

In [ ]:
# 7b(save 6b)
from pyspark.sql import functions as F
import time

# Save each severity count batch to Parquet
def save_severity_count_to_parquet(batch_df, batch_id):
    (
        batch_df
        .withColumn("batch_id", F.lit(batch_id))
        .write
        .mode("append")
        .parquet("./parquet_output/6b_severity_count")
    )

# Save the full updated severity count table every 10 seconds
severity_count_parquet_query = (
    severity_count_df
    .writeStream
    .outputMode("complete")
    .foreachBatch(save_severity_count_to_parquet)
    .option("checkpointLocation", f"./checkpoint/parquet_6b_severity_count_{int(time.time())}")
    .trigger(processingTime="10 seconds")
    .start()
)

In [ ]:
# 7c(save 6c)

# Save each district severity batch to Parquet
def save_district_severity_to_parquet(batch_df, batch_id):
    (
        batch_df
        .withColumn("batch_id", F.lit(batch_id))
        .write
        .mode("append")
        .parquet("./parquet_output/6c_district_severity")
    )

# Save each district severity batch to Parquet
district_severity_parquet_query = (
    district_severity_df
    .writeStream
    .outputMode("complete")
    .foreachBatch(save_district_severity_to_parquet)
    .option("checkpointLocation", f"./checkpoint/parquet_6c_district_severity_{int(time.time())}")
    .trigger(processingTime="30 seconds")
    .start()
)

8. Read the Parquet files from task 7 as data streams and send them to Kafka topics with appropriate names.  
(Note: You shall read the parquet files as a streaming data frame and send messages to the Kafka topic when new data appears in the parquet file.)

In [ ]:
# Stream 1
from pyspark.sql import functions as F
import time

# Read schema from parquet files
schema_6a = high_severity_df.schema

# Read parquet as streaming dataframe
parquet_6a_stream = (
    spark.readStream
    .schema(schema_6a)
    .parquet("./parquet_output/6a_high_severity")
)

# Convert rows to Kafka key-value format
kafka_6a_df = (
    parquet_6a_stream
    .select(
        F.col("collision_index").cast("string").alias("key"),
        F.to_json(F.struct("*")).alias("value")
    )
)

# Send to Kafka topic
parquet_6a_to_kafka_query = (
    kafka_6a_df
    .writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "a2b_6a_high_severity")
    .option("checkpointLocation", f"./checkpoint/8a_parquet_to_kafka")
    .outputMode("append")
    .start()
)

In [ ]:
# Stream 2

# Read schema from parquet files
schema_6b = severity_count_df.schema

# Read parquet as streaming dataframe
parquet_6b_stream = (
    spark.readStream
    .schema(schema_6b)
    .parquet("./parquet_output/6b_severity_count")
)

# Convert rows to Kafka key-value format
kafka_6b_df = (
    parquet_6b_stream
    .select(
        F.col("severity_rating").cast("string").alias("key"),
        F.to_json(F.struct("*")).alias("value")
    )
)
# Send to Kafka topic
parquet_6b_to_kafka_query = (
    kafka_6b_df
    .writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "a2b_6b_severity_count")
    .option("checkpointLocation", f"./checkpoint/8b_parquet_to_kafka")
    .outputMode("append")
    .start()
)

In [ ]:
# Stream 3

# Read schema from parquet files
schema_6c = district_severity_df.schema

# Read parquet as streaming dataframe
parquet_6c_stream = (
    spark.readStream
    .schema(schema_6c)
    .parquet("./parquet_output/6c_district_severity")
)

# Convert rows to Kafka key-value format
kafka_6c_df = (
    parquet_6c_stream
    .select(
        F.col("area").cast("string").alias("key"),
        F.to_json(F.struct("*")).alias("value")
    )
)
# Send to Kafka topic
parquet_6c_to_kafka_query = (
    kafka_6c_df
    .writeStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "kafka:9092")
    .option("topic", "a2b_6c_district_severity")
    .option("checkpointLocation", f"./checkpoint/8c_parquet_to_kafka")
    .outputMode("append")
    .start()
)

In [ ]:
from IPython.display import clear_output
import time

start_time = time.time()

while True:
    clear_output(wait=True)

    elapsed = int(time.time() - start_time)

    print(f"Real-time Accident Severity Dashboard")
    print(f"Elapsed time: {elapsed} seconds")

    # 6a show every 5 seconds
    print("\n High-severity accidents (>7) every 5 seconds")
    spark.sql("""
        SELECT *
        FROM high_severity_table
        ORDER BY accident_time DESC
    """).show(20, truncate=False)

    # 6b show every 10 seconds
    print("\n Total number of accidents for each severity every 10 seconds")
    spark.sql("""
        SELECT severity_rating, count AS total_accidents
        FROM severity_count_table
        ORDER BY severity_rating
    """).show(20, truncate=False)

    # 6c show every 30 seconds
    print("\n The total number of low, medium and high severity accidents for each local district every 30 seconds")
    spark.sql("""
        SELECT *
        FROM district_severity_table
    """).show(50, truncate=False)

    time.sleep(5)